In [1]:
!pip install -q datasets huggingface_hub scikit-learn pandas

In [2]:
import pandas as pd

from datasets import load_dataset
from huggingface_hub import login
from getpass import getpass

HF_TOKEN = getpass("Enter your Hugging Face token: ")
login(token=HF_TOKEN)

print("Logged in.")

Enter your Hugging Face token: ··········
Logged in.


In [3]:
GPT_OSS_REPO_ID = (
    "dinushiTJ/"
    "nz_research_commons_gpt_oss_cross_provider_results_1200"
)

gpt_dataset = load_dataset(
    GPT_OSS_REPO_ID,
    split="train",
)

df = gpt_dataset.to_pandas()

print("Rows loaded:", len(df))
print("Columns:")
print(df.columns.tolist())

README.md:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/7.68M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1200 [00:00<?, ? examples/s]

Rows loaded: 1200
Columns:
['title', 'authors', 'subjects', 'abstract', 'text', 'record_id_hash', 'prompt', 'token_count', 'classification_label', 'classification_reason', 'year', 'classification_confidence', 'embedding_text', 'embedding_token_count', 'conversations', 'label_text', 'row_index', 'gpt_oss_model', 'gpt_oss_reasoning_effort', 'gpt_oss_label', 'gpt_oss_confidence', 'gpt_oss_reason', 'gpt_oss_raw_output', 'gpt_oss_error', 'seconds']


In [4]:
GEMINI_LABEL_COL = "label_text"
GPT_OSS_LABEL_COL = "gpt_oss_label"

print(
    "Gemini label column exists:",
    GEMINI_LABEL_COL in df.columns,
)

print(
    "GPT-OSS label column exists:",
    GPT_OSS_LABEL_COL in df.columns,
)

print("\nGemini labels:")
print(df[GEMINI_LABEL_COL].value_counts(dropna=False))

print("\nGPT-OSS labels:")
print(df[GPT_OSS_LABEL_COL].value_counts(dropna=False))

Gemini label column exists: True
GPT-OSS label column exists: True

Gemini labels:
label_text
non_maori_origin    600
maori_origin        600
Name: count, dtype: int64

GPT-OSS labels:
gpt_oss_label
non_maori_origin    755
maori_origin        445
Name: count, dtype: int64


In [5]:
VALID_LABELS = [
    "maori_origin",
    "non_maori_origin",
]

comparison_df = df[
    df[GEMINI_LABEL_COL].isin(VALID_LABELS)
    & df[GPT_OSS_LABEL_COL].isin(VALID_LABELS)
].copy()

comparison_df["labels_match"] = (
    comparison_df[GEMINI_LABEL_COL]
    == comparison_df[GPT_OSS_LABEL_COL]
)

total_compared = len(comparison_df)
matching_count = int(
    comparison_df["labels_match"].sum()
)
different_count = (
    total_compared - matching_count
)

agreement_rate = (
    matching_count / total_compared
    if total_compared > 0
    else 0
)

print("Total records loaded:", len(df))
print("Total records compared:", total_compared)
print("Matching labels:", matching_count)
print("Different labels:", different_count)
print("Agreement rate:", f"{agreement_rate:.2%}")

Total records loaded: 1200
Total records compared: 1200
Matching labels: 1045
Different labels: 155
Agreement rate: 87.08%


In [6]:
confusion_matrix = pd.crosstab(
    comparison_df[GEMINI_LABEL_COL],
    comparison_df[GPT_OSS_LABEL_COL],
    rownames=["Gemini label"],
    colnames=["GPT-OSS label"],
    margins=True,
)

display(confusion_matrix)

GPT-OSS label,maori_origin,non_maori_origin,All
Gemini label,,,
maori_origin,445,155,600
non_maori_origin,0,600,600
All,445,755,1200


In [7]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    cohen_kappa_score,
)

y_true = comparison_df[GEMINI_LABEL_COL]
y_pred = comparison_df[GPT_OSS_LABEL_COL]

accuracy = accuracy_score(
    y_true,
    y_pred,
)

precision = precision_score(
    y_true,
    y_pred,
    pos_label="maori_origin",
    zero_division=0,
)

recall = recall_score(
    y_true,
    y_pred,
    pos_label="maori_origin",
    zero_division=0,
)

f1 = f1_score(
    y_true,
    y_pred,
    pos_label="maori_origin",
    zero_division=0,
)

kappa = cohen_kappa_score(
    y_true,
    y_pred,
)

print("Accuracy:", f"{accuracy:.2%}")
print(
    "Precision — Māori origin:",
    f"{precision:.2%}",
)
print(
    "Recall — Māori origin:",
    f"{recall:.2%}",
)
print(
    "F1 score — Māori origin:",
    f"{f1:.2%}",
)
print("Cohen's kappa:", round(kappa, 4))

print("\nClassification report:\n")

print(
    classification_report(
        y_true,
        y_pred,
        labels=VALID_LABELS,
        zero_division=0,
    )
)

Accuracy: 87.08%
Precision — Māori origin: 100.00%
Recall — Māori origin: 74.17%
F1 score — Māori origin: 85.17%
Cohen's kappa: 0.7417

Classification report:

                  precision    recall  f1-score   support

    maori_origin       1.00      0.74      0.85       600
non_maori_origin       0.79      1.00      0.89       600

        accuracy                           0.87      1200
       macro avg       0.90      0.87      0.87      1200
    weighted avg       0.90      0.87      0.87      1200



In [8]:
disagreement_df = comparison_df[
    ~comparison_df["labels_match"]
].copy()

print(
    "Number of disagreements:",
    len(disagreement_df),
)

columns_to_show = [
    "row_index",
    "record_id_hash",
    "title",
    GEMINI_LABEL_COL,
    GPT_OSS_LABEL_COL,
    "gpt_oss_confidence",
    "gpt_oss_reason",
]

columns_to_show = [
    column
    for column in columns_to_show
    if column in disagreement_df.columns
]

display(
    disagreement_df[
        columns_to_show
    ]
)

Number of disagreements: 155


,row_index,record_id_hash,title,label_text,gpt_oss_label,gpt_oss_confidence,gpt_oss_reason
28,41,1748aee60bceb198399d93c93a4c0ff6ca2e555f9704fb...,Indigenous Rights Hollow Rights?,maori_origin,non_maori_origin,NaN,Fallback extraction from raw GPT-OSS output.
32,45,2aaa287a6a432683d2dbe0082ee269ca1fdec901643625...,Low copper levels measured in the aortic wall ...,maori_origin,non_maori_origin,NaN,Fallback extraction from raw GPT-OSS output.
46,67,95fa4e08eeca81b7295f6f0454a8954c4ad25e1eb76098...,Secondary school subject choice: an ethnograph...,maori_origin,non_maori_origin,NaN,Fallback extraction from raw GPT-OSS output.
85,117,3a58d90aaa4d2bf3825fe109fcdb14d0401467138c39a9...,Puna,maori_origin,non_maori_origin,NaN,Fallback extraction from raw GPT-OSS output.
163,217,f9f7645f6977d30e5ecd8d2f1a3de11458b99054395a27...,"The Wairua of Self: A Path of Love, Simplicity...",maori_origin,non_maori_origin,0.95,The research focuses on the concept of love an...
...,...,...,...,...,...,...,...
1181,1135,19b4942b4edbfd5b30a501a106e935a488c2ce2ab2d35e...,Cultural justice and ethics: From within,maori_origin,non_maori_origin,0.85,The paper discusses cultural justice and ethic...
1185,1160,47356381825a4e731603554df8edb108bc93da4e958fa4...,Approaches to benefit sharing,maori_origin,non_maori_origin,0.92,The paper discusses benefit sharing for Indige...
1186,1161,30d3eee7f0cc8df1daa315035ca924ab7318dc33d87f4d...,Personality and Representation in Environmenta...,maori_origin,non_maori_origin,0.95,The paper focuses on environmental law concept...
1189,1174,98711904d733ba10b9b7a05765c331d74e9545236d40dd...,Mainstreaming managed retreat in Aotearoa New ...,maori_origin,non_maori_origin,0.85,The paper discusses managed retreat in New Zea...
